In [ ]:
import torch
import json
import os
import hashlib
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import clip
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import yaml
from src.features.extraction import extract_and_cache
from transformers import AutoModelForCausalLM, AutoTokenizer
import re
import numpy as np
from scipy.optimize import minimize

from src.data import (
    NegationJSONDataset, 
    COCOValLlamaDataset, 
    NegRefCOCOgDataset, 
    VALSEDataset
)

from src.features import FeatureCache, extract_and_cache
from src.llm import LocalQwenClient, SubQueryExtractor

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

Use the following filepath:
1. NegRefCOCOg -> ./data/NegRefCOCOg.json
2. VALSE -> ./data/existence.json
3. Gemini (for training steering linear classifier) -> ./data/gemini_negation_dataset2.json

In [ ]:
dataset = NegRefCOCOgDataset("./data/NegRefCOCOg.json", max_samples=7000)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

In [ ]:
#Change the base model and configuration of CLIP
#Also change the dataset name in the extract_and_cache function to match the dataset you are using
base_config = {"type": "vanilla", "arch": "ViT-B/32", "layer": -1}
base_model, _ = clip.load("ViT-B/32", device=device)
base_features = extract_and_cache(base_model, loader, clip.tokenize, "Baseline_CLIP", base_config, dataset="COCOValLlama", device=device)

In [ ]:
#Run this cell if embeddings for all 12 layers of CLIP  not cached. This will cache the features for all 12 layers and save them in the cache directory. You can comment out this cell after running it once to avoid redundant caching in future runs.
layers_to_save = range(1, 13) 
for l in layers_to_save:
    current_cfg = {
        "type": "vanilla", 
        "arch": "ViT-B/32", 
        "layer": l
    }
    
    variant_name = "Baseline_CLIP"
    
    print(f"\n>>> Extracting features for Layer {l}...")
    
    extract_and_cache(
        model=base_model, 
        dataloader=loader, 
        dataset = "GEMINI_NEGATION_2",
        tokenizer=clip.tokenize, 
        model_variant_name=variant_name, 
        config=current_cfg,
        device="cuda"
    )

print("\n All 12 layers have been cached successfully.")

Train a linear classifier to classify positive and negative text and hence learn a negation direction.

In [ ]:
dataset = NegRefCOCOgDataset("./data/gemini_negation_dataset2.json", max_samples=7000)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

In [ ]:
layers_to_save = range(1, 13) 
for l in layers_to_save:
    current_cfg = {
        "type": "vanilla", 
        "arch": "ViT-B/32", 
        "layer": l
    }
    
    variant_name = "NG_CLIP"
    
    print(f"\n>>> Extracting features for Layer {l}...")
    
    extract_and_cache(
        model=base_model, 
        dataloader=loader, 
        dataset = "GEMINI_NEGATION_2",
        tokenizer=clip.tokenize, 
        model_variant_name=variant_name, 
        config=current_cfg,
        device="cuda"
    )

print("\n All 12 layers have been cached successfully.")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import  os
import json

def train_binary_negation_classifier(z_pos, z_neg, config):
    device = config.get('device', 'cuda')
    epochs = config.get('epochs', 150)
    lr = config.get('lr', 1e-3)
    batch_size = config.get('batch_size', 64)
    val_split = config.get('val_split', 0.2)
    
    # 1. Prepare Data
    z_pos = z_pos.float().cpu() 
    z_neg = z_neg.float().cpu()

    num_pairs = len(z_pos)

    # FIX: Split by pairs to prevent data leakage!
    indices = torch.randperm(num_pairs)
    val_size = int(num_pairs * val_split)
    
    val_indices = indices[:val_size]
    train_indices = indices[val_size:]
    
    # Slice the paired data
    train_z_pos, train_z_neg = z_pos[train_indices], z_neg[train_indices]
    val_z_pos, val_z_neg = z_pos[val_indices], z_neg[val_indices]
    
    # Now concatenate positive and negative samples for train and val respectively
    X_train = torch.cat([train_z_pos, train_z_neg], dim=0)
    y_train = torch.cat([torch.zeros(len(train_z_pos)), torch.ones(len(train_z_neg))], dim=0)
    
    X_val = torch.cat([val_z_pos, val_z_neg], dim=0)
    y_val = torch.cat([torch.zeros(len(val_z_pos)), torch.ones(len(val_z_neg))], dim=0)
    
    # Shuffle the resulting datasets so 0s and 1s aren't clumped
    train_ds = TensorDataset(X_train, y_train)
    val_ds = TensorDataset(X_val, y_val)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    
    # 2. Model Setup
    # Use X_train to get the feature dimension
    classifier = nn.Linear(X_train.shape[1], 1).to(device).float()
    nn.init.normal_(classifier.weight, mean=0, std=0.01)
    nn.init.zeros_(classifier.bias)

    # Using Adam Optimizer
    optimizer = optim.Adam(classifier.parameters(), lr=lr, weight_decay=1e-5)
    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )
    
    criterion = nn.BCEWithLogitsLoss()
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'lrs': []
    }

    best_val_acc = 0.0
    best_w = None

    for epoch in range(epochs):
        # --- Training ---
        classifier.train()
        train_loss, train_correct = 0, 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = classifier(batch_X).squeeze()
            loss = criterion(outputs, batch_y)
            
            if torch.isnan(loss):
                print(f"NaN detected at epoch {epoch+1}. Stopping.")
                return None, history
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(classifier.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            train_correct += (preds == batch_y).sum().item()
            
        # --- Validation ---
        classifier.eval()
        val_loss, val_correct = 0, 0
        with torch.no_grad():
            for v_batch_X, v_batch_y in val_loader:
                v_batch_X, v_batch_y = v_batch_X.to(device), v_batch_y.to(device)
                v_outputs = classifier(v_batch_X).squeeze()
                v_loss = criterion(v_outputs, v_batch_y)
                val_loss += v_loss.item()
                v_preds = (torch.sigmoid(v_outputs) > 0.5).float()
                val_correct += (v_preds == v_batch_y).sum().item()

        current_val_acc = val_correct / len(X_val)
        avg_val_loss = val_loss / len(val_loader)
        
        scheduler.step(avg_val_loss)

        # Save weights from the best-performing epoch
        if current_val_acc > best_val_acc:
            best_val_acc = current_val_acc
            best_w = classifier.weight.data.detach().clone()

        # Record History
        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(train_correct / len(X_train))
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(current_val_acc)
        history['lrs'].append(optimizer.param_groups[0]['lr'])

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:03d} | Train Acc: {history['train_acc'][-1]:.4f} | Loss: {history['train_loss'][-1]:.5f} | Val Acc: {current_val_acc:.4f} | Loss: {history['val_loss'][-1]:.5f} | LR: {history['lrs'][-1]:.1e}")

    # Return Best Direction Vector
    if best_w is None:
        return None, history
        
    W_dir = F.normalize(best_w.float(), p=2, dim=-1)
    return W_dir, history



def steer_embeddings(h_l, W_dir, alpha):
    """
    h_l: Original hidden states [N, D]
    W_dir: Normalized classifier weights [1, D]
    alpha: Steering hyperparameter
    """
    norm_h = torch.norm(h_l, p=2, dim=-1, keepdim=True) # ||h_l||
    # Steering formula
    h_steered = (1 - alpha) * h_l + alpha * W_dir * norm_h
    return h_steered

In [ ]:
def run_paper_negation_experiment(config):
    torch.manual_seed(config['seed'])
    cache_manager = FeatureCache()
    device = config['device']
    
    # 1. Load Data
    pos_path = cache_manager.get_cache_path(config['pos_variant'], config['dataset'], config['pos_config'])
    neg_path = cache_manager.get_cache_path(config['neg_variant'], config['dataset'], config['neg_config'])
    pos_data, neg_data = cache_manager.load(pos_path), cache_manager.load(neg_path)
    
    z_pos, z_neg = pos_data['pos_text'], neg_data['neg_text']

    # 2. Train/Test Split (Classifier only sees Train part)
    indices = torch.randperm(z_pos.size(0))
    train_size = int(config['split_ratio'] * z_pos.size(0))
    train_idx, test_idx = indices[:train_size], indices[train_size:]
    
    # 3. Learn Direction (Returns Training & Validation history)
    W_dir, history = train_binary_negation_classifier(
        z_pos[train_idx], z_neg[train_idx], config
    )

    # 4. Final Evaluation (On the absolute Test Set)
    with torch.no_grad():
        test_p = z_pos[test_idx].to(device).float()
        test_n = z_neg[test_idx].to(device).float()
        
        # A. Calculate Test Accuracy of the Classifier
        # This confirms if negation is linearly separable on unseen data
        test_X = torch.cat([test_p, test_n], dim=0)
        test_y = torch.cat([torch.zeros(len(test_p)), torch.ones(len(test_n))], dim=0).to(device)
        
        # Reconstruct classifier logic for a single pass
        test_logits = (test_X @ W_dir.T).squeeze() # Bias omitted as W_dir is the normalized direction
        test_preds = (test_logits > 0).float() # Using 0 as threshold for normalized direction
        test_acc = (test_preds == test_y).sum().item() / len(test_y)

        # B. Steering Evaluation (Equation 1)
        steered_p = steer_embeddings(test_p, W_dir, config['alpha'])
        
        # Final Similarities
        p_n, n_n, s_p_n = F.normalize(test_p, p=2), F.normalize(test_n, p=2), F.normalize(steered_p, p=2)
        baseline_sim = torch.cosine_similarity(p_n, n_n).mean().item()
        final_sim = torch.cosine_similarity(s_p_n, n_n).mean().item()

    # 5. Reporting
    print(f"\n" + "="*30)
    print(f"NEGATION EXPERIMENT: {config['pos_variant']}")
    print(f"="*30)
    print(f"Classifier Training Acc:   {history['train_acc'][-1]:.2%}")
    print(f"Classifier Validation Acc: {history['val_acc'][-1]:.2%}")
    print(f"Classifier Test Acc:       {test_acc:.2%}")
    print(f"-"*30)
    print(f"Baseline Cosine Sim:       {baseline_sim:.4f}")
    print(f"Steered Cosine Sim:        {final_sim:.4f}")
    print(f"Total Gain:                {final_sim - baseline_sim:.4f}")
    print(f"="*30)

    output = {
        "W_dir": W_dir.cpu(),
        "history": history,
        "test_acc": test_acc,
        "baseline_sim": baseline_sim,
        "result_sim": final_sim,
        "gain": final_sim - baseline_sim
    }

    results_dir = "learned_vectors"
    os.makedirs(results_dir, exist_ok=True) 
    
    config_dict = {
        "dataset": config['dataset'],
        "pos_variant": config['pos_variant'],
        "neg_variant": config['neg_variant'],
        "pos_config": config['pos_config'],
        "neg_config": config['neg_config'],
    }

    # 🔹 Step 2: convert to stable string
    config_str = json.dumps(config_dict, sort_keys=True)

    # 🔹 Step 3: hash it
    hash_id = hashlib.sha1(config_str.encode()).hexdigest()[:8]

    # 🔹 Optional: readable prefix (safe)
    safe_name = f"{config['dataset']}_{config['pos_variant']}_{config['neg_variant']}".replace("/", "-")

    # 🔹 Final filename
    save_filename = f"{safe_name}_{hash_id}_negation_vector.pt"
    save_path = os.path.join(results_dir, save_filename)

    # 🔹 Save tensor
    torch.save(output, save_path)

    # 🔹 Save config for reproducibility
    with open(os.path.join(results_dir, f"{safe_name}_{hash_id}_config.json"), "w") as f:
        f.write(config_str)

    print(f"Learned vector and metrics saved to: {save_path}")

    return output

In [ ]:
base_config = {"type": "vanilla", "arch": "ViT-B/32", "layer": 4}

# If you want to compare against your DEO features, use that config
# Otherwise, you can compare Baseline Pos vs Baseline Neg
exp_config = {
    'dataset': "GEMINI_NEGATION_2",       
    'pos_variant': 'NG_CLIP',    # Must match the name used in extract_and_cache
    'neg_variant': 'NG_CLIP',    # Can be DEO_CLIP if you cached that,,
    'pos_config': base_config,
    'neg_config': base_config,
    'split_ratio': 0.8,                # Train/Test split for the classifier
    'val_split': 0.2,                  # Internal validation for the classifier
    'alpha': 0.5,                      # Steering strength (from the paper)
    'lr': 0.01,                        # Classifier learning rate
    'epochs': 40,                      # Classifier training epochs
    'batch_size': 64,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42
}
run_paper_negation_experiment(exp_config)